# Notebook 04: Multi-LLM Backend

## Learning Objectives

In this notebook, you will learn:
1. **Factory Pattern** — One function creates different objects based on configuration
2. **Polymorphism** — Different classes, same interface (`BaseChatModel.invoke()`)
3. **Dependency Inversion** — Depend on abstractions, not concrete implementations
4. **Local vs Cloud LLMs** — Trade-offs between privacy, cost, quality, and latency

## Supported Providers

| Provider | Type | API Key? | Use Case |
|----------|------|----------|----------|
| **OpenAI** | Cloud | Yes (`OPENAI_API_KEY`) | Best quality, easiest setup |
| **Qwen** (通义千问) | Cloud | Yes (`DASHSCOPE_API_KEY`) | Chinese language tasks, Alibaba ecosystem |
| **Ollama** | Local | No | Privacy, offline, free after setup |

## 1. Setup

Make sure you have the required packages installed:
```bash
pip install -r requirements.txt
```

For Ollama, also install and start the Ollama server:
```bash
# Install Ollama: https://ollama.ai
ollama pull qwen2.5:7b    # Download a model
ollama serve               # Start the server
```

In [19]:
import os
import sys
from dotenv import load_dotenv

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

# Load environment variables
load_dotenv()

print(f"Current LLM_PROVIDER: {os.getenv('LLM_PROVIDER', 'openai (default)')}")

Current LLM_PROVIDER: openai


## 2. Understanding the Factory Pattern

The **Factory Pattern** is a design pattern where a single function creates different
objects based on configuration — the caller doesn't need to know the concrete class.

```
.env (LLM_PROVIDER=openai/qwen/ollama)
         |
         v
    get_llm()  ← Factory Function
    /    |    \
   v     v     v
ChatOpenAI  ChatTongyi  ChatOllama
   \     |     /
    v    v    v
  BaseChatModel.invoke()  ← Unified interface!
```

Let's look at how it works:

In [20]:
from src.llm import get_llm, get_llm_info

# Check current configuration
info = get_llm_info()
print("=== LLM Configuration ===")
print(f"Provider: {info['provider']}")
print(f"Model:    {info['model']}")
print(f"Status:   {info['status']}")
print(f"Detail:   {info['detail']}")

=== LLM Configuration ===
Provider: openai
Model:    gpt-4o-mini
Status:   ok
Detail:   API Key: sk-proj-...KpYA


In [21]:
# Create an LLM instance using the factory
llm = get_llm(temperature=0.0)

# The type depends on LLM_PROVIDER, but the interface is the same!
print(f"LLM type: {type(llm).__name__}")
print(f"Is BaseChatModel? {hasattr(llm, 'invoke')}")

# Test it with a simple question
response = llm.invoke("Say hello in one sentence.")
print(f"\nResponse: {response.content}")

LLM type: ChatOpenAI
Is BaseChatModel? True

Response: Hello! How can I assist you today?


## 3. Switching Providers at Runtime

You can switch providers by changing the environment variable.
This is useful for testing or comparing different LLMs.

**Important:** In production, set `LLM_PROVIDER` in `.env`.
Here we change it temporarily for demonstration.

In [22]:
# Helper: test a provider (set env var, create LLM, invoke)
def test_provider(provider_name: str, test_prompt: str = "What is 2+2? Reply in one sentence."):
    """Test a specific LLM provider."""
    original = os.getenv("LLM_PROVIDER")
    try:
        os.environ["LLM_PROVIDER"] = provider_name
        info = get_llm_info()
        print(f"\n{'='*50}")
        print(f"Provider: {info['provider'].upper()} | Model: {info['model']}")
        print(f"Status: {info['status']} | {info['detail']}")
        print(f"{'='*50}")

        if info['status'] != 'ok':
            print(f"⚠️  Skipping — {info['detail']}")
            return None

        llm = get_llm(temperature=0.0)
        print(f"LLM class: {type(llm).__name__}")

        response = llm.invoke(test_prompt)
        print(f"Prompt: {test_prompt}")
        print(f"Response: {response.content}")
        return response.content

    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        # Restore original setting
        if original:
            os.environ["LLM_PROVIDER"] = original
        elif "LLM_PROVIDER" in os.environ:
            del os.environ["LLM_PROVIDER"]


# Test the currently configured provider
current = os.getenv("LLM_PROVIDER", "openai")
test_provider(current)


Provider: OPENAI | Model: gpt-4o-mini
Status: ok | API Key: sk-proj-...KpYA
LLM class: ChatOpenAI
Prompt: What is 2+2? Reply in one sentence.
Response: 2 + 2 equals 4.


'2 + 2 equals 4.'

## 4. Multi-Provider Comparison

If you have multiple providers configured, you can compare their responses.
Uncomment the providers you have set up:

In [23]:
# Compare providers on the same smart-home prompt
test_prompt = (
    "You are a smart home assistant. The user says: "
    "'Make the living room cozy for movie night'. "
    "Briefly describe what you would do (2-3 sentences)."
)

results = {}

# Test available providers (comment out any you don't have configured)
for provider in ["openai", "qwen", "ollama"]:
    result = test_provider(provider, test_prompt)
    if result:
        results[provider] = result

# Summary
print("\n" + "="*60)
print("COMPARISON SUMMARY")
print("="*60)
for provider, response in results.items():
    print(f"\n[{provider.upper()}]:")
    print(f"  {response[:200]}..." if len(response) > 200 else f"  {response}")


Provider: OPENAI | Model: gpt-4o-mini
Status: ok | API Key: sk-proj-...KpYA
LLM class: ChatOpenAI
Prompt: You are a smart home assistant. The user says: 'Make the living room cozy for movie night'. Briefly describe what you would do (2-3 sentences).
Response: I would dim the lights to a warm, soft glow and adjust the temperature to a comfortable level. Then, I would play some soft background music while fluffing the cushions on the couch and setting up any blankets for added comfort. Finally, I would prepare the popcorn and ensure the TV is ready for your movie selection.

Provider: QWEN | Model: qwen-turbo
Status: error | DASHSCOPE_API_KEY not set
⚠️  Skipping — DASHSCOPE_API_KEY not set

Provider: OLLAMA | Model: qwen2.5:3b
Status: ok | Server: http://localhost:11434
LLM class: ChatOllama
Prompt: You are a smart home assistant. The user says: 'Make the living room cozy for movie night'. Briefly describe what you would do (2-3 sentences).
Response: For a cozy movie night in the livin

## 5. Running the Full Agent with Different Providers

The Smart Home Agent uses `get_llm()` internally — so switching
the provider automatically affects the entire agent pipeline.

**Note:** This requires Neo4j to be running with seed data loaded.

In [24]:
# Test the full agent with the current provider
try:
    from src.agent import SmartHomeAgent

    info = get_llm_info()
    print(f"Running agent with: {info['provider'].upper()} ({info['model']})")
    print("=" * 50)

    agent = SmartHomeAgent(debug=True)
    response, trace = agent.run_with_trace("Turn on the living room lights")

    print(f"\nResponse: {response}")
    print(f"\nReasoning Trace:")
    for i, step in enumerate(trace, 1):
        print(f"  {i}. {step}")

except Exception as e:
    print(f"Could not run full agent: {e}")
    print("Make sure Neo4j is running and seed data is loaded.")

Running agent with: OPENAI (gpt-4o-mini)

Response: I'm turning on the living room lights for you! I'll activate both the ceiling light and the floor lamp to brighten up the space. Enjoy the cozy atmosphere! If you need anything else, just let me know!

Reasoning Trace:
  1. Parsed intent: goal='turn on lights', rooms=['living room']
  2. Retrieved context using strategies: ['room_context']
  3. Context is sufficient to generate plan
  4. Generated plan with 2 actions
  5. Validation: passed
  6. Generated user response


## 6. Key Takeaways

### Design Patterns Used

| Pattern | Where | Why |
|---------|-------|-----|
| **Factory** | `get_llm()` | Create different LLM objects from config |
| **Polymorphism** | `BaseChatModel.invoke()` | All LLMs share the same interface |
| **Lazy Import** | `_create_*_llm()` | Only import libraries when needed |
| **Dependency Inversion** | Agent nodes | Depend on abstract `BaseChatModel`, not `ChatOpenAI` |

### Cloud vs Local LLMs

| Aspect | Cloud (OpenAI/Qwen) | Local (Ollama) |
|--------|--------------------|-----------------|
| **Quality** | Higher (larger models) | Lower (smaller models) |
| **Latency** | Network dependent | Local, potentially faster |
| **Cost** | Pay per token | Free (hardware cost) |
| **Privacy** | Data sent to cloud | Data stays local |
| **Setup** | API key only | Install Ollama + download model |
| **Availability** | Requires internet | Works offline |